<a href="https://colab.research.google.com/github/jac0bmath3w/rail-safety-ai/blob/main/notebooks/03_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/jac0bmath3w/rail-safety-ai.git

In [ ]:
# Install Unsloth and dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo
!pip install --no-deps "xformers<0.0.29" "trl<0.13.0" peft accelerate bitsandbytes
!pip install pypdf
!pip install langchain
!pip install langchain-community
!pip install langchain-huggingface
!pip install llama-index
!pip install -U langchain-text-splitters
!pip install chromadb
!pip install -U bitsandbytes>=0.46.1
!pip install --upgrade transformers
!pip install sentencepiece tiktoken

In [ ]:
import os, sys
import torch
from google.colab import drive
from google.colab import userdata
src_path = os.path.join(os.getcwd(), 'rail-safety-ai', 'src')
sys.path.append(src_path)
from vector_store import RailVectorVault
from embed import RailEmbedder
from generator import RailDataGenerator

# Mount Drive to access Vector DB and save the model
drive.mount('/content/drive')
api_key = userdata.get('GEMINI_API_KEY')
DB_PATH = "/content/drive/MyDrive/rail_ai_project/vector_db"
TRAIN_DATA_PATH = "/content/rail-safety-ai/data/training/rail_dataset.jsonl"

In [ ]:
#
# os.path.dirname("/content/rail-safety-ai/data/training/rail_dataset.jsonl")
os.makedirs(os.path.dirname(TRAIN_DATA_PATH), exist_ok=True)

In [ ]:
import importlib
import generator
importlib.reload(generator)
from generator import RailDataGenerator

In [ ]:
# Initialize the Librarian (Vault) and the Teacher (Generator)
embedder = RailEmbedder(model_name='BAAI/bge-base-en-v1.5')
vault = RailVectorVault(embedder_instance=embedder, db_path=DB_PATH)


model_id = "gemini-2.5-flash"

url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_id}:generateContent"

generator = RailDataGenerator(api_url = url, #"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-preview-09-2025:generateContent",
                              api_key = api_key,
                              vault_instance=vault)



In [ ]:
# Generate the synthetic dataset
# 1 sample took about 30 seconds
res_dir = generator.create_dataset(num_samples=100, output_path=TRAIN_DATA_PATH)

In [ ]:
!wc -l /content/rail-safety-ai/data/training/rail_dataset.jsonl

In [ ]:
import json
gold_standard_example = {
    'question': 'A maintenance technician inspects an active grade crossing warning system and notes that one of the gate arm lights is illuminated but appears significantly dimmer than the other lights on the same gate arm. During daylight hours, this particular light is difficult to discern from the prescribed approach distance of 200 feet, although it is clearly visible at night. The light unit itself and its wiring are observed to be securely fastened to the gate arm.\n\nBased on FRA regulations under Section 234-29 (implied 234.219), which specific defect classification(s) apply to this situation, and what is the overarching intent of this section that is being violated by this condition?',
    'thinking': 'PHASE 1: CONTEXTUAL AUDIT\n1. **Source Document**: FRA-Signal_Train_Control-2012.pdf, Page 124, Section 234-29 (which corresponds to 234.219 for defect codes).\n2. **Core Subject**: Maintenance, visibility, and securement of gate arm lights and wires at active warning systems.\n3. **Key Requirements/Provisions**: Each gate arm light must be properly visible to approaching highway users (and pedestrians), lights and wires must be securely fastened, and maintenance must adhere to design specifications.\n4. **Specific Defect Classifications**: A1 (burned out/missing light), A2 (defective/not visible/missing light unit), A3 (light unit not secured), A4 (wires not secured), A5 (not maintained per design specifications).\n5. **Intent Statement**: "The intent of this section is that lights and light wires shall be maintained in accordance with design specifications." This is critical.\n6. **Question Goal**: Create a challenging scenario requiring differentiation between similar defect codes and linking to the section\'s overarching intent.\n\nPHASE 2: EVIDENCE MAPPING\n1. **Analyze Scenario Elements**: \n * "illuminated but appears significantly dimmer": Rules out A1 (\'burned out or missing\' literally).\n * "difficult to discern from the prescribed approach distance of 200 feet during daylight hours": Directly violates the requirement for lights to be "properly visible to approaching highway users." This strongly points to A2 (\'not visible\') and A5 (\'not maintained per design specifications\').\n * "clearly visible at night": Confirms the light is functional but performance is compromised under specific conditions (daylight), reinforcing a maintenance/design specification issue.\n * "light unit itself and its wiring are observed to be securely fastened": Rules out A3 (\'Light unit not securely fastened\') and A4 (\'Light wires not securely fastened\').\n\n2. **Map Scenario to Defect Codes**: \n * **234.219.A1 (Gate arm light burned out or missing)**: *Not applicable* because the light is illuminated.\n * **234.219.A2 (Gate arm light unit defective, not visible, or missing)**: *Applicable*. The light is "not visible" from the required distance during daylight, and its dimness suggests the unit is "defective" in its performance.\n * **234.219.A3 (Light unit not securely fastened)**: *Not applicable* as it\'s stated to be securely fastened.\n * **234.219.A4 (Light wires not securely fastened)**: *Not applicable* as wiring is stated to be securely fastened.\n * **234.219.A5 (Gate arm light unit not maintained per design specifications)**: *Applicable*. The diminished brightness and failure to be visible under daylight conditions indicate a failure to meet design specifications for performance and maintenance. This is further supported by the section\'s stated intent.\n\n3. **Identify Overarching Intent**: The excerpt explicitly states: "The intent of this section is that lights and light wires shall be maintained in accordance with design specifications." The scenario directly demonstrates a failure to meet this, as the light\'s performance (visibility) has degraded from its specified standard.\n\nPHASE 3: SYNTHESIS\n* **Applicable Defects**: 234.219.A2 (due to being not visible/defective) and 234.219.A5 (due to not meeting design specs for visibility/brightness). \n* **Non-Applicable Defects**: 234.219.A1, A3, A4 (with clear justifications).\n* **Overarching Intent Violation**: Failure to maintain lights in accordance with design specifications, which directly impacts their intended visibility for safety.\n\nPHASE 4: VERIFICATION\n* **Completeness**: All parts of the question (specific classifications, overarching intent) are addressed.\n* **Accuracy**: The classifications and reasoning directly align with the provided text. The distinction between A1 and A2/A5 is carefully made.\n* **Clarity**: The answer is structured logically, explaining why each code applies or doesn\'t apply.\n* **Challenging Aspect**: The question forces the student to go beyond simple identification (e.g., \'burned out\') to interpret \'not visible\' and \'not maintained per design specifications\' in a nuanced scenario where the light is technically \'on\'. It also requires understanding the explicit \'intent\' statement.',
    'answer': 'The specific defect classifications applicable to this situation are:\n\n1. **234.219.A2: Gate arm light unit defective, not visible, or missing.**\n * The scenario clearly states that the light, despite being illuminated, is "difficult to discern from the prescribed approach distance of 200 feet during daylight hours." This directly meets the "not visible" criterion as required for approaching highway users. Furthermore, the description of it being "significantly dimmer" strongly indicates that the light unit itself is "defective" in its performance, as its output has degraded below acceptable operating levels.\n\n2. **234.219.A5: Gate arm light unit not maintained per design specifications.**\n * The observed diminished brightness and subsequent lack of visibility under daylight conditions demonstrate a failure to maintain the gate arm light unit to its intended design specifications for luminance and conspicuity. The proper functioning and visibility of the light are fundamental design requirements, and their degradation constitutes a failure in maintenance according to those specifications.\n\nThe following defect classifications are *not* applicable:\n\n* **234.219.A1 (Gate arm light burned out or missing)**: This is not applicable because the light is explicitly stated to be "illuminated," meaning it is neither burned out nor physically missing.\n* **234.219.A3 (Light unit not securely fastened to gate arm)**: This is not applicable as the scenario states the "light unit itself... [is] observed to be securely fastened."\n* **234.219.A4 (Light wires not securely fastened to gate arm)**: This is not applicable as the scenario states the "wiring [is] observed to be securely fastened."\n\nThe **overarching intent** of this section that is being violated is precisely stated in the manual excerpt: **"The intent of this section is that lights and light wires shall be maintained in accordance with design specifications."** The observed condition—a significantly dimmer light that is not adequately visible in daylight—directly contravenes this intent by demonstrating a failure to maintain the gate arm light unit\'s performance to its designed safety and operational standards for ensuring proper warning to approaching highway users.'
}

with open(TRAIN_DATA_PATH, 'a') as f:
    f.write(json.dumps(gold_standard_example) + '\n')

In [ ]:
!tail -n 1 /content/rail-safety-ai/data/training/rail_dataset.jsonl

In [ ]:
from unsloth import FastLanguageModel

model_id = "unsloth/Llama-3.2-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = 2048,
    load_in_4bit = True,
)

# Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: The "Staff" choice for balancing capacity vs. memory
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized to 0 for Unsloth
    bias = "none",    # Optimized to "none" for Unsloth
    use_gradient_checkpointing = "unsloth",
)


In [ ]:
import json
from datasets import Dataset

def formatting_prompts_func(examples):
    instructions = examples["question"]
    thoughts = examples["thinking"]
    answers = examples["answer"]
    texts = []

    for instruction, thought, answer in zip(instructions, thoughts, answers):
        # We wrap the thinking and answer together as the 'Assistant' response
        full_response = f"[THINKING PROCESS]\n{thought}\n\n[ANSWER]\n{answer}"

        messages = [
            {"role": "system", "content": "You are a Senior FRA Safety Consultant. Use a 4-Phase Thinking Process."},
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": full_response},
        ]

        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return { "text" : texts, }





In [ ]:
# Load the JSONL that was generated

import pandas as pd
df = pd.read_json(TRAIN_DATA_PATH, lines=True)
base_dataset = Dataset.from_pandas(df)
text_dataset = base_dataset.map(formatting_prompts_func, batched = True,
                                remove_columns = base_dataset.column_names)

In [ ]:
# def tokenize_fn(example):
#   out = tokenizer(
#       example['text'],
#       trucation=True,
#       padding=False,
#       max_length=2048,
#   )
#   return {"input_ids": out["input_ids"], "attention_mask": out["attention_mask"]}

# tokenized_dataset = text_dataset.map(tokenize_fn, batched=True, remove_columns="text")

# print(tokenized_dataset.column_names)
# print(tokenized_dataset[0].keys())

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling

class DropTextCollator:
  def __init__(self, tokenizer):
    self.base = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

  def __call__(self, features):
    cleaned = []
    for f in features:
      f = dict(f)
      f.pop('text', None)
      cleaned.append(f)
    return self.base(cleaned)

args = SFTConfig(
    per_device_train_batch_size=1, # Reduced to 1
    gradient_accumulation_steps=8,
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    output_dir="outputs",
    report_to="none",
    push_to_hub=False,
    max_seq_length=1024,
    remove_unused_columns=True,
    # dataset_kwargs={"skip_prepare_dataset": True},
    optim = "adamw_8bit",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=text_dataset,   # only input_ids + attention_mask
    data_collator = DropTextCollator(tokenizer),
    args = args
)

print(trainer.train_dataset.column_names)

In [ ]:
trainer.train()

In [ ]:
save_path = "/content/drive/MyDrive/rail_ai_project/rail_safety_adapters"
os.makedirs(save_path, exist_ok=True)

# Standard saving method for PEFT/LoRA models
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)